# 3.3 SES Prototype — SLA Breach Proxy (Reproducible GitHub Version)
**Purpose:** Generate SLA breach risk artefacts consumed by the Streamlit dashboard, using the existing repo structure.

**Inputs (repo):**
- `data/processed/sla_breach_events.csv` (if present)
- `data/processed/sla_test_scores.csv` (if present)
- `data/processed/sla_flags_test.csv` (optional)

**Outputs (repo):**
- `reports/figures/sla_event_shap_values.csv`
- `reports/figures/sla_event_heatmap.png`
- `reports/figures/sla_continuous_heatmap.png`
- `data/processed/sla_breach_events.csv` (ensured)
- `data/processed/sla_test_scores.csv` (ensured)

Notes:
- Designed to run from `notebooks/` inside the GitHub repo.
- If ground-truth breach labels are not available, this notebook produces a conservative proxy dataset suitable for dashboard demo.
- SHAP explanations are generated with a **surrogate model** (Random Forest) trained to approximate the SLA risk score.


In [ ]:
# ============================================================
# 0) Imports
# ============================================================
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

import matplotlib.pyplot as plt
import shap


In [ ]:
# ============================================================
# 1) Resolve repo paths (DO NOT change repo tree)
# ============================================================
HERE = Path.cwd().resolve()
REPO = HERE
while not (REPO / "app.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "app.py").exists(), f"Repo root not found. Current working dir: {HERE}"

DATA_DIR = REPO / "data" / "processed"
FIG_DIR  = REPO / "reports" / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

IN_BREACH = DATA_DIR / "sla_breach_events.csv"
IN_SCORES = DATA_DIR / "sla_test_scores.csv"

print("REPO    :", REPO)
print("DATA_DIR:", DATA_DIR)
print("FIG_DIR :", FIG_DIR)
print("IN_BREACH:", IN_BREACH)
print("IN_SCORES:", IN_SCORES)


In [ ]:
# ============================================================
# 2) Load or create SLA proxy datasets
# ============================================================
def ensure_sla_scores() -> pd.DataFrame:
    # Preferred: existing scores
    if IN_SCORES.exists():
        df_scores = pd.read_csv(IN_SCORES)
        # Normalise column names expected by app.py
        if "time" not in df_scores.columns:
            for cand in ("timestamp", "t", "start"):
                if cand in df_scores.columns:
                    df_scores = df_scores.rename(columns={cand: "time"})
                    break
        df_scores["time"] = pd.to_datetime(df_scores["time"], errors="coerce", utc=True)

        # Ensure risk_score exists
        if "risk_score" not in df_scores.columns:
            for cand in ("anomaly_score", "proba_raw", "proba", "score", "sla_risk"):
                if cand in df_scores.columns:
                    df_scores["risk_score"] = pd.to_numeric(df_scores[cand], errors="coerce")
                    break
        if "risk_score" not in df_scores.columns:
            df_scores["risk_score"] = 0.0

        df_scores["risk_score"] = df_scores["risk_score"].fillna(method="ffill").fillna(0.0)
        return df_scores[["time", "risk_score"]].sort_values("time").reset_index(drop=True)

    # Fallback: generate a minimal synthetic proxy series for demo
    n = 1500
    ts = pd.date_range("2021-11-01", periods=n, freq="min", tz="UTC")
    rng = np.random.default_rng(42)
    base = rng.normal(0.15, 0.03, size=n).clip(0, 1)
    for center in (400, 850, 1200):
        ramp = np.exp(-((np.arange(n) - center) ** 2) / (2 * 40 ** 2))
        base = np.clip(base + 0.5 * ramp, 0, 1)

    return pd.DataFrame({"time": ts, "risk_score": base})

def ensure_sla_breaches(df_scores: pd.DataFrame) -> pd.DataFrame:
    if IN_BREACH.exists():
        df_b = pd.read_csv(IN_BREACH)

        if "start" not in df_b.columns:
            for cand in ("t_start", "time_start", "time"):
                if cand in df_b.columns:
                    df_b = df_b.rename(columns={cand: "start"})
                    break
        if "end" not in df_b.columns:
            for cand in ("t_end", "time_end"):
                if cand in df_b.columns:
                    df_b = df_b.rename(columns={cand: "end"})
                    break

        df_b["start"] = pd.to_datetime(df_b["start"], errors="coerce", utc=True)
        df_b["end"] = pd.to_datetime(df_b["end"], errors="coerce", utc=True)

        if "duration_s" not in df_b.columns:
            df_b["duration_s"] = (df_b["end"] - df_b["start"]).dt.total_seconds()

        if "severity" not in df_b.columns:
            df_b["severity"] = "high"
        if "breach_flag" not in df_b.columns:
            df_b["breach_flag"] = 1

        return df_b.sort_values("start").reset_index(drop=True)

    thr = float(np.quantile(df_scores["risk_score"].values, 0.995))
    g = df_scores.copy()
    g["flag"] = g["risk_score"] >= thr
    g["grp"] = (g["flag"] != g["flag"].shift(1)).cumsum()

    rows = []
    for _, chunk in g.groupby("grp"):
        if not bool(chunk["flag"].iloc[0]):
            continue
        start = chunk["time"].iloc[0]
        end = chunk["time"].iloc[-1]
        rows.append({
            "start": start,
            "end": end,
            "duration_s": float((end - start).total_seconds()),
            "severity": "high",
            "breach_flag": 1,
        })

    return pd.DataFrame(rows).sort_values("start").reset_index(drop=True)

sla_scores = ensure_sla_scores()
sla_breaches = ensure_sla_breaches(sla_scores)

# Persist ensured datasets
sla_scores.to_csv(IN_SCORES, index=False)
sla_breaches.to_csv(IN_BREACH, index=False)

print("Saved/ensured:", IN_SCORES, "rows:", len(sla_scores))
print("Saved/ensured:", IN_BREACH, "rows:", len(sla_breaches))

sla_scores.head(), sla_breaches.head()


In [ ]:
# ============================================================
# 3) Build feature set for SHAP surrogate
# ============================================================
df_feat = sla_scores.copy().sort_values("time").reset_index(drop=True)

df_feat["risk_lag_1"] = df_feat["risk_score"].shift(1)
df_feat["risk_lag_5"] = df_feat["risk_score"].shift(5)
df_feat["risk_roll_mean_15"] = df_feat["risk_score"].rolling(15, min_periods=1).mean()
df_feat["risk_roll_std_15"] = df_feat["risk_score"].rolling(15, min_periods=1).std().fillna(0.0)
df_feat["risk_slope_10"] = df_feat["risk_score"].diff(10) / 10.0

df_feat = df_feat.replace([np.inf, -np.inf], np.nan).fillna(method="ffill").fillna(0.0)

feature_cols = ["risk_score", "risk_lag_1", "risk_lag_5", "risk_roll_mean_15", "risk_roll_std_15", "risk_slope_10"]
X = df_feat[feature_cols].values
y = df_feat["risk_score"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature matrix:", X_scaled.shape)


In [ ]:
# ============================================================
# 4) Train surrogate + compute SHAP event matrix
# ============================================================
rf = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_scaled, y)

explainer = shap.TreeExplainer(rf)

# Choose a window around the first breach (or global peak)
if not sla_breaches.empty:
    t0 = pd.to_datetime(sla_breaches["start"].iloc[0], utc=True)
    t1 = pd.to_datetime(sla_breaches["end"].iloc[0], utc=True)
    pad = pd.Timedelta(minutes=30)
    w_start, w_end = t0 - pad, t1 + pad
    mask = (df_feat["time"] >= w_start) & (df_feat["time"] <= w_end)
    idxs = np.where(mask.values)[0]
else:
    peak = int(np.argmax(df_feat["risk_score"].values))
    idxs = np.arange(max(0, peak - 40), min(len(df_feat), peak + 41))

max_steps = 40
if len(idxs) > max_steps:
    idxs = np.linspace(idxs.min(), idxs.max(), max_steps).round().astype(int)

X_win = X_scaled[idxs]  # (time_steps, n_features)

shap_vals = np.asarray(explainer.shap_values(X_win))  # (time_steps, n_features)
shap_matrix = shap_vals.T  # (features, time)
time_labels = [f"t{i}" for i in range(shap_matrix.shape[1])]

def save_shap_matrix_csv(out_csv: Path, shap_matrix_2d: np.ndarray, feature_names: list[str], time_labels: list[str]) -> None:
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(shap_matrix_2d, index=feature_names, columns=time_labels).to_csv(out_csv)
    print("Saved SHAP matrix CSV ->", out_csv)

OUT_SHAP = FIG_DIR / "sla_event_shap_values.csv"
save_shap_matrix_csv(OUT_SHAP, shap_matrix, feature_cols, time_labels)


In [ ]:
# ============================================================
# 5) Save heatmap PNGs (event + continuous)
# ============================================================
OUT_EVENT_PNG = FIG_DIR / "sla_event_heatmap.png"
plt.figure(figsize=(10, 5))
plt.imshow(shap_matrix, aspect="auto")
plt.yticks(np.arange(len(feature_cols)), feature_cols, fontsize=8)
plt.xticks(np.arange(len(time_labels)), time_labels, rotation=90, fontsize=7)
plt.title("SLA – SHAP heatmap around one breach window (surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_EVENT_PNG, dpi=200, bbox_inches="tight")
plt.close()
print("Saved:", OUT_EVENT_PNG)

OUT_CONT_PNG = FIG_DIR / "sla_continuous_heatmap.png"
rng = np.random.default_rng(42)
n_windows = min(180, len(X_scaled))
sel = np.sort(rng.choice(len(X_scaled), size=n_windows, replace=False))
X_sub = X_scaled[sel]
shap_sub = np.asarray(explainer.shap_values(X_sub))  # (n_windows, n_features)

mean_abs = np.mean(np.abs(shap_sub), axis=0)
top_n = min(len(feature_cols), 6)
top_idx = np.argsort(mean_abs)[::-1][:top_n]

cont = shap_sub[:, top_idx].T
cont_feat = [feature_cols[i] for i in top_idx]
cont_time = [f"w{i}" for i in range(n_windows)]

plt.figure(figsize=(10, 5))
plt.imshow(cont, aspect="auto")
plt.yticks(np.arange(len(cont_feat)), cont_feat, fontsize=8)
plt.xticks(np.arange(len(cont_time))[::15], cont_time[::15], rotation=90, fontsize=7)
plt.title("SLA – Continuous SHAP overview (surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_CONT_PNG, dpi=200, bbox_inches="tight")
plt.close()
print("Saved:", OUT_CONT_PNG)


## Completion check
Verify these outputs exist:
- `data/processed/sla_test_scores.csv`
- `data/processed/sla_breach_events.csv`
- `reports/figures/sla_event_shap_values.csv`
- `reports/figures/sla_event_heatmap.png`
- `reports/figures/sla_continuous_heatmap.png`
